# Federal Grants & Contracts Finder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/federal-grants-contracts.ipynb)

**What:** Open and forecasted federal grant opportunities from Grants.gov plus contract and grant awards from USAspending.gov, in one unified schema: deadlines, ceilings, amounts, agency, NAICS/PSC, recipient, and a link to the official record. No API key. $0.002 per opportunity, $0.0015 per award.

**Why:** run the [`federal-grants-contracts`](https://apify.com/transparent_meteorite/federal-grants-contracts) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/federal-grants-contracts) for every option.

In [2]:
run_input = {   'keywords': ['cybersecurity'],
    'mode': 'both',
    'maxItems': 5,
    'awardKinds': ['contracts'],
    'awardsLookbackDays': 365,
    'minAwardAmount': 1000000}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/federal-grants-contracts').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> Status: RUNNING, Message: 


[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> 2026-10-13T22:17:43.659Z ACTOR: Pulling container image of build iih20MPv7dWkPVN91 from registry.


[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> 2026-10-13T22:17:43.663Z ACTOR: Creating container.


[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> 2026-10-13T22:17:43.728Z ACTOR: Starting container.


[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> 2026-10-13T22:17:43.729Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> 2026-10-13T22:17:47.248Z Done in 2s. 5 opportunities, 0 awards; charged 5 + 0; 0 warnings.


[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> 2026-10-13T22:17:47.248Z INFO  [Status message]: Done: 5 items


[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> 2026-10-13T22:17:47.310Z Thanks for using Federal Grants & Contracts Finder. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/LDaZtDD0K0OzaeoBk/issues) with the run URL and we will reply there.


[apify.federal-grants-contracts runId:yEui0kem9fCbOMduV] -> Status: SUCCEEDED, Message: Done: 5 items


Run status: SUCCEEDED | items: 5


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["source", "recordType", "title", "agency", "status", "postedDate", "closeDate", "amountCeiling", "amount", "recipient"] if c in df.columns]
df[SHOW]

,source,recordType,title,agency,status,postedDate,closeDate,amountCeiling,amount,recipient
0,grants.gov,opportunity,Artificial Intelligence and Cybersecurity Education Innovation and...,U.S. National Science Foundation,posted,2026-02-04,2027-04-05,2500000.0,None,None
1,grants.gov,opportunity,2023 Emerging Voices Small Grants Program,Department of State,forecasted,2023-04-24,NaN,NaN,None,None
2,grants.gov,opportunity,Cybersecurity Innovation for Cyberinfrastructure (CICI),U.S. National Science Foundation,posted,2025-01-04,2027-01-20,1200000.0,None,None
3,grants.gov,opportunity,Advisory Support for Establishing a National Cybersecurity Operati...,Department of State,posted,2026-09-10,2026-10-16,13912185.0,None,None
4,grants.gov,opportunity,Department of War Cyber Service Academy (DoW CSA),Department of Defense,posted,2025-01-23,2030-01-22,20000000.0,None,None


## 5. A quick analysis

In [5]:
df['best_amount']=pd.to_numeric(df['amount'].fillna(df['amountCeiling']),errors='coerce')
print(df.groupby('recordType')['best_amount'].agg(['count','sum']))
df.sort_values('best_amount',ascending=False)[['title','agency','best_amount']].head(5)

             count         sum
recordType                    
opportunity      4  37612185.0


,title,agency,best_amount
4,Department of War Cyber Service Academy (DoW CSA),Department of Defense,20000000.0
3,Advisory Support for Establishing a National Cybersecurity Operati...,Department of State,13912185.0
0,Artificial Intelligence and Cybersecurity Education Innovation and...,U.S. National Science Foundation,2500000.0
2,Cybersecurity Innovation for Cyberinfrastructure (CICI),U.S. National Science Foundation,1200000.0
1,2023 Emerging Voices Small Grants Program,Department of State,NaN


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [federal-grants-contracts on the Apify Store](https://apify.com/transparent_meteorite/federal-grants-contracts).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.